In [ ]:
# MIT License

# Copyright (c) 2024 Multifidelity Roofline Analysis

# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:

# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.

# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

%load_ext autoreload
%autoreload 2

In [ ]:
from GenZ import chunked_moddeling
from GenZ import get_model_df, System, get_summary_table, simplify_df, get_configs, ParallelismConfig, get_runtime_breakdown
from GenZ.Models.get_language_model import create_full_chunked_model, create_full_decode_model
from GenZ.Models.attention import mha_flash_attention_chunked
from GenZ.utils.plot_rooflines import display_df
import os
import pandas as pd


In [ ]:
get_configs('meta-llama/meta-llama-3.1-405b')

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

# def interactive_chunked_model_runtime(input_tokens, output_tokens, output_batch):
input_tokens = 1024
output_tokens = 1024
TP = 4
chunk_size = 512
system = System(frequency=1000 , flops=4500, off_chip_mem_size=(192*1024), compute_efficiency=1, memory_efficiency=1,
                offchip_mem_bw=8000, bits='int8', external_mem_bw=128, interchip_link_bw=900, interchip_link_latency=0.25, num_nodes=4)
runtime_plot_df = pd.DataFrame(columns=['Batching Type', 'Layer Name', 'Runtime', 'Chunk Size', 'Decode Batch'])
for chunk_size in [512,  2048]:
    for output_batch in [1, 16, 64]:
        # model = create_full_decode_model(name='gpt-3', input_sequence_length = output_tokens, tensor_parallel=TP)
        # model_df = get_model_df(model, system, batch_size=output_batch,  model_characterstics = False)
        # summary_table = get_summary_table(model_df, model_characterstics = False)
        # runtime_df = get_runtime_breakdown(model_df)
        # # display(model_df)
        # # display(summary_table)
        # df = pd.DataFrame(
        #     [   ['Continous', 'Embedding',   runtime_df.Embedding, chunk_size, output_batch],
        #         ['Continous', 'Collective',  runtime_df.Collective, chunk_size, output_batch],
        #         ['Continous', 'LA_layers',   runtime_df.LA_layers, chunk_size, output_batch],
        #         ['Continous', 'QKVO_layers', runtime_df.QKVO_layers, chunk_size, output_batch],
        #         ['Continous', 'FFN_layers',  runtime_df.FFN_layers, chunk_size, output_batch]],
        #     columns=['Batching Type', 'Layer Name', 'Runtime','Chunk Size', 'Decode Batch'])
        # runtime_plot_df = pd.concat([runtime_plot_df, df])
        # model = create_full_chunked_model(name='gpt-3',
        #                                     prefill_kv_sizes= [(input_tokens, chunk_size-output_batch)],
        #                                     decode_kv_sizes = [output_tokens] * output_batch, tensor_parallel=TP)
        # model_df = get_model_df(model, system, batch_size=1,  model_characterstics = False)
        # summary_table = get_summary_table(model_df, model_characterstics = False)
        # runtime_df = get_runtime_breakdown(model_df)
        # display(summary_table)
        # display(model_df)
        chunked_output = chunked_moddeling(model='gpt-3', 
                        prefill_kv_sizes = [(input_tokens, chunk_size-output_batch)],
                        system_name=system, 
                        decode_kv_sizes = [output_tokens] * output_batch, tensor_parallel=TP)
        runtime_df = chunked_output['Runtime_breakdown']
        df = pd.DataFrame(
                [['Chunked', 'Embedding',   runtime_df.Embedding, chunk_size, output_batch],
                ['Chunked', 'Collective',  runtime_df.Collective, chunk_size, output_batch],
                ['Chunked', 'LA',   runtime_df.LA_layers, chunk_size, output_batch], 
                ['Chunked', 'QKVO', runtime_df.QKVO_layers, chunk_size, output_batch],
                ['Chunked', 'FFN',  runtime_df.FFN_layers, chunk_size, output_batch]],
            columns=['Batching Type', 'Layer Name', 'Runtime','Chunk Size', 'Decode Batch'])
        runtime_plot_df = pd.concat([runtime_plot_df, df])


    # return fig
    # display(runtime_plot_df)

# input_tokens_slider = widgets.BoundedIntText(min=1, max=20480, step=10, value=1024, description='Input Tokens:')
# output_tokens_slider = widgets.BoundedIntText(min=1, max=20480, step=10, value=1024, description='Output Tokens:')
# batch_slider = widgets.BoundedIntText(min=1, max=512, step=8, value=8, description='Output Batch:')

# interactive_chunked_model_runtime(input_tokens=1024, output_tokens=1024, output_batch=8)

In [ ]:
import plotly.express as px
runtime_plot_df['Decode Batch'] = runtime_plot_df['Decode Batch'].astype(str)



fig = px.bar(runtime_plot_df, x='Decode Batch', y='Runtime', color='Layer Name', facet_row='Chunk Size',
            pattern_shape="Batching Type", barmode='relative',
            title='Runtime Breakdown by Chunk Size, Input Tokens = {}, Output Tokens = {}'.format(input_tokens, output_tokens))

fig.update_layout(
    autosize=False,
    width=1000,
    height=800,
)
fig.show()

In [ ]:
runtime_plot_df

In [ ]:
import matplotlib.pyplot as plt

runtime_plot_df['Decode Batch'] = runtime_plot_df['Decode Batch'].astype(str)
runtime_plot_df['Decode Batch'] = pd.Categorical(runtime_plot_df['Decode Batch'], categories=runtime_plot_df['Decode Batch'].unique(), ordered=True)

fig, ax = plt.subplots(figsize=(10, 4), ncols=2)



for i, (key, grp) in enumerate(runtime_plot_df[runtime_plot_df['Batching Type'] == 'Chunked'].groupby(['Chunk Size'])):
    # print(grp.pivot(index=['Decode Batch'], columns='Layer Name', values='Runtime'))
    grp.pivot(index=['Decode Batch'], columns='Layer Name', values='Runtime').plot(kind='bar', stacked=True, ax=ax[i], label=key)

    # ax[i].set_title('Runtime Breakdown by Chunk Size, Input Tokens = {}, Output Tokens = {}, Chunk Size= {}'.format(input_tokens, output_tokens, key[0]))
    ax[i].set_title('Chunk Size= {}'.format(key[0]))
    ax[i].set_ylabel('Latency (ms)')
    ax[i].xaxis.set_tick_params(rotation=0)
    ax[i].tick_params(axis='both', which='major', labelsize=24)

    ax[i].title.set_size(30)
    ax[i].xaxis.label.set_size(24)
    ax[i].yaxis.label.set_size(24)
    # ax[i].xaxis.tick_size = 32
    ax[i].legend(loc='upper left', fontsize=18)
    # ax[i].legend = 2
    ax[i].set_ylim(0, 50)
    if i == 1:
        ax[i].legend().set_visible(False)
        ax[i].set_ylabel('')
    ax[i].set_xlabel('Batch')
plt.savefig('chunked_runtime_breakdown.pdf', format='pdf', dpi=300)
# plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
system = System(frequency=1000 , flops=4500, off_chip_mem_size=(192*1024), compute_efficiency=1, memory_efficiency=1,
                offchip_mem_bw=8000, bits='int8', external_mem_bw=128, interchip_link_bw=900, interchip_link_latency=0.25, num_nodes=4)
chunked_output = chunked_moddeling(model='deepseek-ai/DeepSeek-V3-Base',
                        prefill_kv_sizes = [(0, 256)],
                        system_name=system, debug=True, bits='int8',
                        decode_kv_sizes = [256, 512] , tensor_parallel=8)

In [ ]:
system = System(frequency=1000 , flops=4500, off_chip_mem_size=(192*1024), compute_efficiency=1, memory_efficiency=1,
                offchip_mem_bw=8000, bits='int8', external_mem_bw=128, interchip_link_bw=900, interchip_link_latency=0.25, num_nodes=4)
chunked_output = chunked_moddeling(model='deepseek-ai/DeepSeek-V3-Base',
                        prefill_kv_sizes = [], model_profilling=True,
                        system_name=system, debug=True, bits='int8',
                        decode_kv_sizes = [1] , tensor_parallel=1)
display(chunked_output[0])
display(chunked_output[1])


In [ ]:
chunked_moddeling(model='gpt-3', 
                        prefill_kv_sizes = [],
                        decode_kv_sizes= [],
                        system_name=system,
                        tensor_parallel=TP, debug=True)

In [ ]:
chunked_moddeling(model='meta-llama/meta-llama-3.1-70b',
                        prefill_kv_sizes = [],
                        decode_kv_sizes= [(1,8,2589, 30)],
                        system_name='H100', system_eff=0.75,
                        tensor_parallel=4, debug=True)['Latency']

In [ ]:
chunked_moddeling(model='meta-llama/meta-llama-3.1-70b',
                        prefill_kv_sizes = [],
                        decode_kv_sizes= [(8,1621)], bits='int8',
                        system_name='H100', system_eff=0.75,
                        tensor_parallel=2, debug=True)['Latency']

In [ ]:
chunked_moddeling(model='meta-llama/meta-llama-3.1-70b',
                        prefill_kv_sizes = [(491,222)],
                        decode_kv_sizes= ([(1, 64, 491, 205)]),
                        system_name='H100', system_eff=0.75,
                        tensor_parallel=2, debug=False)['Latency']

In [ ]:
chunked_moddeling(model='meta-llama/meta-llama-3.1-70b',
                        prefill_kv_sizes = [(491,222)],
                        decode_kv_sizes= ([(64, 491+ 205)]),
                        system_name='H100', system_eff=0.75,
                        tensor_parallel=2, debug=True)['Latency']